# 13f CLIP-SENet CityFlowV2 Fine-tune

Fine-tunes the v6 CLIP-SENet VeRi-776 checkpoint on CityFlowV2 GT vehicle crops extracted at runtime from raw videos. Crop extraction follows the working `09g_resnet101ibn_dmt` CityFlow recipe using `thanhnguyenle/data-aicity-2023-track-2`; downstream MTMC evaluation is intentionally left to the 13d-style fusion notebooks.

In [ ]:
%pip install -q --upgrade --index-url https://download.pytorch.org/whl/cu124 torch==2.4.1+cu124 torchvision==0.19.1+cu124
%pip install -q timm open_clip_torch pretrainedmodels loguru

In [ ]:
import json
import math
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from loguru import logger
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Sampler

SEED = 3407
WORKING = Path('/kaggle/working')
OUTPUT_DIR = WORKING
CKPT_DIR = WORKING / 'checkpoints'
FINAL_WEIGHTS_PATH = WORKING / 'vehicle_clip_senet_cityflow_ft.pth'
FINAL_METADATA_PATH = WORKING / 'vehicle_clip_senet_cityflow_ft_metadata.json'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'seed': SEED,
    'num_classes': None,
    'image_size': (320, 320),
    'batch_p': 8,
    'batch_k': 8,
    'batch_size': 64,
    'accum_steps': 2,
    'effective_batch_size': 128,
    'epochs': 12,
    'warmup_epochs': 2,
    'lr': 1e-4,
    'weight_decay': 5e-4,
    'label_smoothing': 0.1,
    'supcon_temperature': 0.07,
    'num_workers': 4,
    'print_every': 25,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [json_ready(item) for item in value]
    if isinstance(value, list):
        return [json_ready(item) for item in value]
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value


def torch_load(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


set_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SESSION_START_TIME = time.time()
print('DEVICE:', DEVICE)
print('torch:', torch.__version__)
print('torchvision:', getattr(T, '__version__', 'unknown'))

In [ ]:
# Reference: notebooks/kaggle/09g_resnet101ibn_dmt/09g_resnet101ibn_dmt.ipynb
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
EXPECTED_CITYFLOW_DATASET_ROOT = KAGGLE_INPUT_ROOT / 'data-aicity-2023-track-2'


def dedupe_paths(paths):
    unique = []
    seen = set()
    for path in paths:
        text = str(path)
        if text in seen:
            continue
        seen.add(text)
        unique.append(path)
    return unique


def find_named_paths(name, want_dir=None, max_results=50):
    if not KAGGLE_INPUT_ROOT.exists():
        return []
    matches = []
    for path in KAGGLE_INPUT_ROOT.rglob(name):
        if want_dir is True and not path.is_dir():
            continue
        if want_dir is False and not path.is_file():
            continue
        matches.append(path)
        if len(matches) >= max_results:
            break
    return sorted(matches)


def resolve_cityflow_root():
    candidate_bases = dedupe_paths(
        [EXPECTED_CITYFLOW_DATASET_ROOT]
        + find_named_paths('data-aicity-2023-track-2', want_dir=True, max_results=20)
    )
    for base in candidate_bases:
        if base.exists():
            print(f'[cityflow] using dataset root: {base}')
            return base

    nested_roots = find_named_paths('AIC22_Track1_MTMC_Tracking', want_dir=True, max_results=20)
    if nested_roots:
        root = nested_roots[0].parent
        print(f'[cityflow] inferred dataset root from nested directory: {root}')
        return root

    raise FileNotFoundError(
        'CityFlowV2 raw dataset not found anywhere under /kaggle/input. '
        'Expected dataset slug: data-aicity-2023-track-2'
    )


def resolve_raw_split_roots(cityflow_root):
    candidates = []
    for subpath in (
        'AIC22_Track1_MTMC_Tracking/train',
        'AIC22_Track1_MTMC_Tracking/validation',
        'train',
        'validation',
    ):
        candidate = cityflow_root / subpath
        if candidate.exists():
            candidates.append(candidate)

    candidates = dedupe_paths(candidates)
    if candidates:
        print('[cityflow] raw split roots:')
        for candidate in candidates:
            print(f'  - {candidate}')
        return candidates

    inferred_roots = []
    for gt_path in find_named_paths('gt.txt', want_dir=False, max_results=200):
        if gt_path.parent.name != 'gt':
            continue
        camera_dir = gt_path.parent.parent
        scene_dir = camera_dir.parent
        if not camera_dir.name.startswith('c') or not scene_dir.name.startswith('S'):
            continue
        if not (camera_dir / 'vdo.avi').exists():
            continue
        inferred_roots.append(scene_dir.parent)

    inferred_roots = dedupe_paths(inferred_roots)
    if inferred_roots:
        print('[cityflow] inferred split roots from gt/video scan:')
        for candidate in inferred_roots:
            print(f'  - {candidate}')
        return inferred_roots

    raise FileNotFoundError('CityFlowV2 raw train/validation roots not found under /kaggle/input.')


CITYFLOW_ROOT = resolve_cityflow_root()
RAW_SPLIT_ROOTS = resolve_raw_split_roots(CITYFLOW_ROOT)

EXTRACTION_CFG = {
    'sample_stride': 5,
    'max_crops_per_id_cam': 24,
    'min_area': 2000,
    'min_bbox_side': 30,
}

CROP_DIR = OUTPUT_DIR / 'cityflowv2_crops'
CROP_DIR.mkdir(parents=True, exist_ok=True)
CROP_MANIFEST_PATH = OUTPUT_DIR / 'cityflowv2_crops_manifest.json'


def camera_key(camera_dir: Path) -> str:
    return f'{camera_dir.parent.name}_{camera_dir.name}'


camera_dirs = sorted(
    camera_dir
    for split_root in RAW_SPLIT_ROOTS
    for camera_dir in split_root.glob('S*/c*')
    if camera_dir.is_dir()
)
if not camera_dirs:
    raise RuntimeError(f'No camera directories found under {RAW_SPLIT_ROOTS}')

CAMERA_DIRS = {camera_key(camera_dir): camera_dir for camera_dir in camera_dirs}
CAMERA_NAMES = sorted(CAMERA_DIRS)


def load_gt_rows(gt_path: Path) -> list[tuple[int, int, int, int, int, int]]:
    rows = []
    with gt_path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            parts = raw_line.strip().split(',')
            if len(parts) < 6:
                continue
            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))
            x = int(float(parts[2]))
            y = int(float(parts[3]))
            w = int(float(parts[4]))
            h = int(float(parts[5]))
            confidence = float(parts[6]) if len(parts) > 6 else 1.0
            if track_id <= 0 or confidence <= 0:
                continue
            rows.append((frame_id, track_id, x, y, w, h))
    return rows


def sample_track_detections(detections):
    ordered = sorted(detections, key=lambda item: item[0])
    sampled = ordered[:: EXTRACTION_CFG['sample_stride']]
    if ordered and sampled and sampled[-1][0] != ordered[-1][0]:
        sampled.append(ordered[-1])
    max_crops = EXTRACTION_CFG['max_crops_per_id_cam']
    if max_crops and len(sampled) > max_crops:
        keep = np.linspace(0, len(sampled) - 1, num=max_crops, dtype=int)
        sampled = [sampled[index] for index in keep.tolist()]
    return sampled


def extract_camera_crops(cam_name: str, cam_dir: Path) -> dict[int, list[dict]]:
    gt_path = cam_dir / 'gt' / 'gt.txt'
    video_path = cam_dir / 'vdo.avi'
    if not gt_path.exists() or not video_path.exists():
        print(f'Skipping {cam_name}: missing {gt_path.name if not gt_path.exists() else video_path.name}')
        return {}

    detections = defaultdict(list)
    for frame_id, track_id, x, y, w, h in load_gt_rows(gt_path):
        if w * h < EXTRACTION_CFG['min_area']:
            continue
        if min(w, h) < EXTRACTION_CFG['min_bbox_side']:
            continue
        detections[track_id].append((frame_id, x, y, w, h))

    sampled_by_frame = defaultdict(list)
    for track_id, track_detections in detections.items():
        for frame_id, x, y, w, h in sample_track_detections(track_detections):
            sampled_by_frame[frame_id].append((track_id, x, y, w, h))

    if not sampled_by_frame:
        print(f'Skipping {cam_name}: no detections after filtering')
        return {}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f'Skipping {cam_name}: failed to open {video_path}')
        return {}

    records_by_track = defaultdict(list)
    written = 0
    for frame_id in sorted(sampled_by_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(frame_id - 1, 0))
        ok, frame = cap.read()
        if not ok:
            continue
        frame_h, frame_w = frame.shape[:2]
        for track_id, x, y, w, h in sampled_by_frame[frame_id]:
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame_w, x + w)
            y2 = min(frame_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            crop_path = CROP_DIR / f'{track_id:04d}_{cam_name}_f{frame_id:06d}.jpg'
            if not crop_path.exists() and not cv2.imwrite(str(crop_path), crop):
                continue
            records_by_track[track_id].append(
                {
                    'path': str(crop_path),
                    'pid': int(track_id),
                    'camname': cam_name,
                    'frame_id': int(frame_id),
                }
            )
            written += 1

    cap.release()
    print(f'{cam_name}: wrote {written} crops across {len(records_by_track)} vehicle IDs')
    return {track_id: sorted(items, key=lambda item: item['frame_id']) for track_id, items in records_by_track.items()}


def manifest_is_usable(payload: dict) -> bool:
    if not CROP_DIR.exists():
        return False
    if payload.get('crop_dir') != str(CROP_DIR):
        return False
    return any(CROP_DIR.glob('*.jpg'))


all_crops = None
if CROP_MANIFEST_PATH.exists():
    with CROP_MANIFEST_PATH.open('r', encoding='utf-8') as handle:
        manifest_payload = json.load(handle)
    if manifest_is_usable(manifest_payload):
        all_crops = {
            int(pid): {
                camname: [dict(record) for record in records]
                for camname, records in camera_records.items()
            }
            for pid, camera_records in manifest_payload['all_crops'].items()
        }
        print(f'Reusing cached crops from {CROP_DIR}')

if all_crops is None:
    for existing_crop in CROP_DIR.glob('*.jpg'):
        existing_crop.unlink()
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam_name in CAMERA_NAMES:
        extracted = extract_camera_crops(cam_name, CAMERA_DIRS[cam_name])
        for pid, records in extracted.items():
            all_crops[pid][cam_name].extend(records)
    all_crops = {
        int(pid): {camname: sorted(records, key=lambda item: item['frame_id']) for camname, records in camera_records.items()}
        for pid, camera_records in all_crops.items()
    }
    with CROP_MANIFEST_PATH.open('w', encoding='utf-8') as handle:
        json.dump(
            {
                'cityflow_root': str(CITYFLOW_ROOT),
                'raw_split_roots': [str(path) for path in RAW_SPLIT_ROOTS],
                'crop_dir': str(CROP_DIR),
                'all_crops': all_crops,
            },
            handle,
            indent=2,
            ensure_ascii=True,
        )

if not all_crops:
    raise RuntimeError('No crops were extracted from CityFlowV2 raw videos')

train_records = []
for pid in sorted(all_crops):
    for camname, records in sorted(all_crops[pid].items()):
        for record in records:
            train_records.append(dict(record))

camname_to_id = {}
for record in train_records:
    if record['camname'] not in camname_to_id:
        camname_to_id[record['camname']] = len(camname_to_id)
    record['camid'] = camname_to_id[record['camname']]

train_pid_map = {pid: index for index, pid in enumerate(sorted({record['pid'] for record in train_records}))}
for record in train_records:
    record['original_pid'] = int(record['pid'])
    record['pid'] = train_pid_map[record['pid']]

CFG['num_classes'] = len(train_pid_map)
if CFG['num_classes'] < CFG['batch_p']:
    raise RuntimeError(f"Need at least {CFG['batch_p']} identities for PK sampling, found {CFG['num_classes']}")

SPLIT_STATS = {
    'cityflow_root': str(CITYFLOW_ROOT),
    'raw_split_roots': [str(path) for path in RAW_SPLIT_ROOTS],
    'crop_dir': str(CROP_DIR),
    'train_images': len(train_records),
    'num_train_ids': CFG['num_classes'],
    'num_cameras': len(camname_to_id),
    'batch_size': CFG['batch_size'],
    'effective_batch_size': CFG['effective_batch_size'],
    'pk_sampler': {'p': CFG['batch_p'], 'k': CFG['batch_k']},
    'extraction': EXTRACTION_CFG,
    'reference_notebook': 'notebooks/kaggle/09g_resnet101ibn_dmt/09g_resnet101ibn_dmt.ipynb',
}
print(json.dumps(SPLIT_STATS, indent=2))

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)


CHECKPOINT_CANDIDATE_NAMES = ('best.pth', 'vehicle_clip_senet_veri776.pth')
PREFERRED_SOURCE_SLUG = '13-clip-senet-train'


def discover_clip_senet_checkpoint() -> Path:
    candidates = []
    for root in sorted(Path('/kaggle/input').rglob('*')):
        if not root.is_dir():
            continue
        for name in CHECKPOINT_CANDIDATE_NAMES:
            candidate = root / name
            if candidate.is_file():
                score = 0
                candidate_str = str(candidate)
                if PREFERRED_SOURCE_SLUG in candidate_str:
                    score += 100
                if 'checkpoints' in candidate.parts:
                    score += 10
                if name == 'best.pth':
                    score += 5
                candidates.append((score, candidate))
    if not candidates:
        raise FileNotFoundError(
            'Could not find CLIP-SENet checkpoint under /kaggle/input. '
            'Expected chained kernel source output from yahiaakhalafallah/13-clip-senet-train.'
        )
    candidates.sort(key=lambda item: (-item[0], str(item[1])))
    print('Checkpoint candidates:')
    for score, path in candidates[:10]:
        print(f'  score={score:3d}  {path}')
    return candidates[0][1]


def extract_state_dict(payload):
    if isinstance(payload, dict) and 'model_state' in payload:
        return payload['model_state'], 'payload:model_state'
    if isinstance(payload, dict) and 'model' in payload and isinstance(payload['model'], dict):
        return payload['model'], 'payload:model'
    if isinstance(payload, dict) and payload and all(hasattr(value, 'shape') for value in payload.values()):
        return payload, 'state_dict'
    raise TypeError(f'Unsupported checkpoint format: {type(payload).__name__}')


CHECKPOINT_PATH = discover_clip_senet_checkpoint()
payload = torch_load(CHECKPOINT_PATH)
state_dict, checkpoint_kind = extract_state_dict(payload)
source_classifier = state_dict.get('classifier.weight')
source_num_classes = int(source_classifier.shape[0]) if source_classifier is not None else None
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('CHECKPOINT_KIND:', checkpoint_kind)
print('SOURCE_NUM_CLASSES:', source_num_classes)
print('CITYFLOW_NUM_CLASSES:', CFG['num_classes'])

model = build_clip_senet(num_classes=CFG['num_classes']).to(DEVICE)
load_state = dict(state_dict)
if source_classifier is not None and source_num_classes != CFG['num_classes']:
    print('Reinitializing classifier.weight for CityFlow class count')
    load_state.pop('classifier.weight', None)

missing_keys, unexpected_keys = model.load_state_dict(load_state, strict=False)
allowed_missing = {'classifier.weight'} if source_num_classes != CFG['num_classes'] else set()
unexpected_keys = list(unexpected_keys)
missing_keys = list(missing_keys)
if set(missing_keys) - allowed_missing or unexpected_keys:
    print('Missing keys:', missing_keys)
    print('Unexpected keys:', unexpected_keys)
    raise RuntimeError('Unexpected checkpoint load mismatch')

model.train()
RUN_INFO = {
    'device': DEVICE,
    'checkpoint_path': str(CHECKPOINT_PATH),
    'checkpoint_kind': checkpoint_kind,
    'source_num_classes': source_num_classes,
    'cityflow_num_classes': CFG['num_classes'],
    'classifier_reinitialized': source_num_classes != CFG['num_classes'],
}
print(json.dumps(RUN_INFO, indent=2))

In [ ]:
class ReIDImageDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(record['pid']), int(record['camid'])


class RandomIdentitySampler(Sampler):
    def __init__(self, records, p, k):
        self.records = records
        self.p = p
        self.k = k
        self.batch_size = p * k
        self.pid_to_indices = defaultdict(list)
        for index, record in enumerate(records):
            self.pid_to_indices[int(record['pid'])].append(index)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        batch_indices = []
        shuffled_pids = self.pids[:]
        random.shuffle(shuffled_pids)
        for pid in shuffled_pids:
            candidates = self.pid_to_indices[pid]
            if len(candidates) >= self.k:
                picked = random.sample(candidates, self.k)
            else:
                picked = random.choices(candidates, k=self.k)
            batch_indices.extend(picked)
            if len(batch_indices) >= self.batch_size:
                yield from batch_indices[: self.batch_size]
                batch_indices = batch_indices[self.batch_size :]

    def __len__(self):
        return len(self.pids) * self.k


height, width = CFG['image_size']
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((height + 16, width + 16), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((height, width)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
    T.RandomErasing(p=0.5, value='random'),
])

train_loader = DataLoader(
    ReIDImageDataset(train_records, train_transform),
    batch_size=CFG['batch_size'],
    sampler=RandomIdentitySampler(train_records, CFG['batch_p'], CFG['batch_k']),
    num_workers=CFG['num_workers'],
    pin_memory=True,
    drop_last=True,
    persistent_workers=CFG['num_workers'] > 0,
)

print('PK sampler:', CFG['batch_p'], 'x', CFG['batch_k'], 'batch_size=', CFG['batch_size'])
print('Training batches per epoch:', len(train_loader))

In [ ]:
class CrossEntropyLabelSmooth(torch.nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.log_softmax = torch.nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.log_softmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_one_hot = (1 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        return (-targets_one_hot * log_probs).mean(0).sum()


class SupConLoss(torch.nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        logits = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)
        logits_mask = torch.ones_like(mask) - torch.eye(mask.shape[0], device=features.device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True).clamp_min(1e-12))
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1).clamp_min(1.0)
        return -mean_log_prob_pos.mean()


def build_scheduler(optimizer):
    def lr_lambda(epoch_idx):
        if epoch_idx < CFG['warmup_epochs']:
            return float(epoch_idx + 1) / float(max(1, CFG['warmup_epochs']))
        progress = float(epoch_idx - CFG['warmup_epochs'] + 1) / float(max(1, CFG['epochs'] - CFG['warmup_epochs']))
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = build_scheduler(optimizer)
scaler = GradScaler(enabled=(DEVICE == 'cuda'))
ce_criterion = CrossEntropyLabelSmooth(num_classes=CFG['num_classes'], epsilon=CFG['label_smoothing']).to(DEVICE)
supcon_criterion = SupConLoss(temperature=CFG['supcon_temperature']).to(DEVICE)
HISTORY = []


def train_one_epoch(epoch_idx: int):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running = {'loss': 0.0, 'ce': 0.0, 'supcon': 0.0, 'steps': 0}
    for step_idx, (images, pids, _camids) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        pids = pids.to(DEVICE, non_blocking=True)
        with autocast(enabled=(DEVICE == 'cuda')):
            features, logits = model(images)
            ce_loss = ce_criterion(logits, pids)
            supcon_loss = supcon_criterion(features, pids)
            total_loss = ce_loss + supcon_loss
            scaled_loss = total_loss / CFG['accum_steps']
        scaler.scale(scaled_loss).backward()
        should_step = ((step_idx + 1) % CFG['accum_steps'] == 0) or (step_idx + 1 == len(train_loader))
        if should_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running['loss'] += float(total_loss.item())
        running['ce'] += float(ce_loss.item())
        running['supcon'] += float(supcon_loss.item())
        running['steps'] += 1
        if step_idx % CFG['print_every'] == 0:
            print(
                f"EPOCH {epoch_idx + 1:02d} STEP {step_idx:04d} "
                f"loss={total_loss.item():.4f} ce={ce_loss.item():.4f} "
                f"supcon={supcon_loss.item():.4f} lr={optimizer.param_groups[0]['lr']:.7f}",
                flush=True,
            )
    return {key: value / max(1, running['steps']) for key, value in running.items() if key != 'steps'}


for epoch_idx in range(CFG['epochs']):
    metrics = train_one_epoch(epoch_idx)
    scheduler.step()
    row = {
        'epoch': epoch_idx + 1,
        'lr': optimizer.param_groups[0]['lr'],
        'train_loss': metrics['loss'],
        'train_ce': metrics['ce'],
        'train_supcon': metrics['supcon'],
    }
    HISTORY.append(row)
    print('EPOCH SUMMARY:', json.dumps(row), flush=True)

print('Training complete')

In [ ]:
final_epoch = len(HISTORY)
checkpoint_payload = {
    'epoch': final_epoch,
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'scheduler_state': scheduler.state_dict(),
    'cfg': json_ready(CFG),
    'history': json_ready(HISTORY),
    'run_info': json_ready(RUN_INFO),
    'split_stats': json_ready(SPLIT_STATS),
}

last_path = CKPT_DIR / 'last.pth'
best_path = CKPT_DIR / 'best.pth'
torch.save(checkpoint_payload, last_path)
torch.save(checkpoint_payload, best_path)
torch.save(checkpoint_payload, FINAL_WEIGHTS_PATH)

metadata = {
    'model_name': 'CLIP-SENet',
    'dataset': 'CityFlowV2 ReID crops',
    'reference_crop_notebook': 'notebooks/kaggle/09g_resnet101ibn_dmt/09g_resnet101ibn_dmt.ipynb',
    'source_checkpoint': RUN_INFO['checkpoint_path'],
    'checkpoint_kind': RUN_INFO['checkpoint_kind'],
    'classifier_reinitialized': RUN_INFO['classifier_reinitialized'],
    'num_classes': CFG['num_classes'],
    'image_size': CFG['image_size'],
    'embed_dim': getattr(model, 'embed_dim', 2048),
    'afem_groups': 32,
    'loaded_resnext_model': getattr(model, 'loaded_resnext_model', 'unknown'),
    'loaded_tinyclip_model': getattr(model, 'loaded_tinyclip_model', 'unknown'),
    'training': {
        'epochs': CFG['epochs'],
        'warmup_epochs': CFG['warmup_epochs'],
        'batch_p': CFG['batch_p'],
        'batch_k': CFG['batch_k'],
        'batch_size': CFG['batch_size'],
        'accum_steps': CFG['accum_steps'],
        'effective_batch_size': CFG['effective_batch_size'],
        'optimizer': 'Adam',
        'lr': CFG['lr'],
        'weight_decay': CFG['weight_decay'],
        'label_smoothing': CFG['label_smoothing'],
        'supcon_temperature': CFG['supcon_temperature'],
        'random_erasing_p': 0.5,
        'amp_fp16': DEVICE == 'cuda',
    },
    'artifacts': {
        'final_weights': str(FINAL_WEIGHTS_PATH),
        'last': str(last_path),
        'best': str(best_path),
    },
    'split_stats': SPLIT_STATS,
    'history': HISTORY,
}
metadata = json_ready(metadata)
with FINAL_METADATA_PATH.open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=True)

print(f'SAVED FINAL CHECKPOINT: {FINAL_WEIGHTS_PATH}')
print(f'SAVED LAST: {last_path}')
print(f'SAVED BEST: {best_path}')
print(f'SAVED METADATA: {FINAL_METADATA_PATH}')
print(json.dumps(metadata, indent=2))